<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/notebooks/04_compile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04. コンパイル — ZK 回路への変換

**前提ファイル**: `network.onnx`, `settings.json`
（`03_calibrate.ipynb` まで実行済み）

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [3]:
import ezkl, os

model_path          = '/content/drive/MyDrive/ezkl/network.onnx'
compiled_model_path = '/content/drive/MyDrive/ezkl/network.compiled'
settings_path       = '/content/drive/MyDrive/ezkl/settings.json'

for f in [model_path, settings_path]:
    assert os.path.exists(f), f"{f} がありません。"

## `compile_circuit` が何をするか

ONNX の計算グラフを **ZK 回路（算術制約の集まり）** に変換します。

RareSkills で学んだ流れがそのまま対応します：
```
ONNX ノード（Conv, Relu, Gemm...）
        ↓ 各操作を算術演算に分解
    R1CS 制約（Aw ∘ Bw = Cw）
        ↓ QAP 変換
   多項式の制約
        ↓ Groth16 / PLONK の回路構造
    network.compiled
```

EZKL がこの変換を全自動で行います。

## 各 op_type が何の制約になるか

| op_type | 制約の種類 | コスト感 |
|---|---|---|
| `Conv` | 多数の乗算制約（チャンネル × カーネルサイズ） | 重い |
| `Relu` | 範囲証明制約（output ≥ 0 を証明） | 中 |
| `Gemm` | 行列積の乗算制約 | 中〜重 |
| `Reshape` | データ移動のみ | 軽い |

`logrows`（回路の行数）はこの制約数の総量によって決まります。

In [4]:
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res == True
print(f"コンパイル成功")
print(f"network.compiled: {os.path.getsize(compiled_model_path):,} bytes")

コンパイル成功
network.compiled: 127,065 bytes


---
次: `05_witness.ipynb` で実際の入力データを使って Witness（計算証拠）を生成します。